<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/1_Diagramas_de_persistencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hopf normal

Este notebook calcula diagramas de persistencia (H₀ y H₁) para el sistema Hopf normal mediante embedding de Takens (d=2, τ=26). Los parámetros clave son: μ ∈ [-1, 1], ruido = 0%, pero ajustable, stride = 2. Los diagramas se generan con Ripser y se visualizan en subplots 3×3. El código asume teaspoon, gtda y ripser instalados.

In [ ]:
############################################################
#     SUBPLOTS DE DIAGRAMAS DE PERSISTENCIA (2D)       #
############################################################

from gtda.time_series import SingleTakensEmbedding
from ripser import ripser
import matplotlib.pyplot as plt
import numpy as np
import teaspoon.MakeData.DynSysLib.DynSysLib as DSL

# --- valores del parámetro mu ---
mu_values = [-1, -0.5, -0.1, -0.08, -0.05, 0, 0.1, 0.5, 1]  # 3x3

embedding_dimension_2D = 2      # Dimensión del embedding
embedding_time_delay_2D = 26    # Tiempo de retardo del embedding
stride_2D = 2

# --- Configuración de Takens 2D ---
embedder_ts_2D = SingleTakensEmbedding(
    parameters_type="fixed",
    time_delay=embedding_time_delay_2D,
    dimension=embedding_dimension_2D,
    stride=stride_2D,
    n_jobs=2
)

# Configuración de estilo
plt.rcParams['figure.figsize'] = (10, 5)

# --- preparar figure ---
fig, axs = plt.subplots(3, 3, figsize=(6, 6.5))
axs = axs.flatten()

ruido = 0.0  # % de ruido

def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
    """
    Agrega ruido gaussiano controlado a una serie temporal.

    Parámetros:
    - ts : array (n_variables, n_samples) o (n_samples,)
    - noise_level : float
    - mode : "absolute" o "relative"
    - seed : int o None (reproducibilidad)

    Retorna:
    - ts_noisy
    """
    if seed is not None:
        np.random.seed(seed)

    ts = np.asarray(ts)

    if mode == "relative":
        scale = noise_level * np.std(ts, axis=-1, keepdims=True)
    else:
        scale = noise_level

    noise = scale * np.random.randn(*ts.shape)
    return ts + noise

for i, mu in enumerate(mu_values):
    # -------- Simular Hopf normal con mu variable ---------
    parameters = [mu, 6]  # (mu, omega)
    t, ts = DSL.DynamicSystems(
        system='hopf_normal',
        dynamic_state=None,
        L=50,
        fs=100,
        SampleSize=2000,
        UserGuide=False,
        parameters=parameters,
        InitialConditions=[0, 1.01]  # [x_0, y_0]
    )

    # ===============================
    # Agregar ruido controlado
    # ===============================
    noise_level = ruido   # % del desvío estándar
    ts = add_noise(ts, noise_level=noise_level, mode="relative", seed=42)

    # -------- Takens embedding 2D --------
    X_emb = embedder_ts_2D.fit_transform(ts[0])

    # -------- Calcular persistencia con Ripser --------
    dgms = ripser(X_emb, maxdim=1)['dgms']

    # -------- Graficar --------
    ax = axs[i]

    ax.xaxis.get_offset_text().set_fontsize(6)
    ax.yaxis.get_offset_text().set_fontsize(6)
    ax.xaxis.get_offset_text().set_alpha(1)
    ax.yaxis.get_offset_text().set_alpha(1)

    colors = ['blue', 'red']
    labels = ['$H_{0}$', '$H_{1}$']
    max_val = 0

    for dim, diagram in enumerate(dgms):
        diagram = diagram[~np.isinf(diagram[:, 1])]
        if diagram.size == 0:
            continue

        ax.scatter(
            diagram[:, 0], diagram[:, 1],
            color=colors[dim],
            label=labels[dim],
            alpha=0.7,
            s=12
        )

        max_val = max(max_val, diagram.max())

    # línea diagonal
    if max_val > 0:
        ax.plot([0, max_val], [0, max_val], 'k--', lw=0.6)

    ax.set_xlabel("Birth", fontsize=6, labelpad=1)
    ax.set_ylabel("Death", fontsize=6, labelpad=1)
    ax.tick_params(axis='both', pad=1, labelsize=6)

    ax.legend(
        fontsize=6,
        loc='lower right',
        borderpad=0.2,
        labelspacing=0.2,
        handlelength=1,
        handletextpad=0.3,
        framealpha=0.6
    )

    ax.set_title(f"$\\mu$ = {mu:.2f}", fontsize=6, pad=1)
    ax.grid(alpha=0.25)
    ax.set_aspect("equal")

plt.tight_layout()
plt.subplots_adjust(hspace=0.00, wspace=0.4, top=0.95, bottom=0.05, left=0.08, right=0.98)

plt.savefig(f"figure_hopf_normal_persistence_diagrams_m={embedding_dimension_2D}_tau={embedding_time_delay_2D}.pdf",
            bbox_inches="tight", dpi=600)
plt.show()

# Lorenz

Este notebook calcula diagramas de persistencia (H₀ y H₁) para el sistema Lorenz con parámetro ρ ∈ [20,30] (20 valores). Usa embedding de Takens (d=2, τ=16, stride=2) y Ripser. Sin ruido añadido. Visualización en subplots 4×5. Requiere gtda, ripser, teaspoon y matplotlib instalados.

In [ ]:
############################################################
#     SUBPLOTS DE DIAGRAMAS DE PERSISTENCIA (2D)       #
############################################################

from gtda.time_series import SingleTakensEmbedding
from ripser import ripser
import matplotlib.pyplot as plt
import numpy as np
import teaspoon.MakeData.DynSysLib.DynSysLib as DSL

# --- valores del parámetro rho ---
rho_values = np.round(np.linspace(20, 30, 20), 10)

embedding_dimension_2D = 2
embedding_time_delay_2D = 16
stride_2D = 2

# --- Configuración de Takens 2D ---
embedder_ts_2D = SingleTakensEmbedding(
    parameters_type="fixed",
    time_delay=embedding_time_delay_2D,
    dimension=embedding_dimension_2D,
    stride=stride_2D,
    n_jobs=2
)

# Configuración de estilo
plt.rcParams['figure.figsize'] = (12, 6)

# --- preparar figure ---
fig, axs = plt.subplots(4, 5, figsize=(13, 10))
axs = axs.flatten()

ruido = 0.0

def add_noise(ts, noise_level=0.0, mode="relative", seed=None):
    if seed is not None:
        np.random.seed(seed)
    ts = np.asarray(ts)
    if mode == "relative":
        scale = noise_level * np.std(ts, axis=-1, keepdims=True)
    else:
        scale = noise_level
    noise = scale * np.random.randn(*ts.shape)
    return ts + noise

for i, rho in enumerate(rho_values):
    # Simular Lorenz con rho variable
    parameters = [rho, 10, 8/3]
    t, ts = DSL.DynamicSystems(
        system='lorenz',
        dynamic_state=None,
        L=100,
        fs=130,
        SampleSize=2000,
        UserGuide=False,
        parameters=parameters,
        InitialConditions=[6.74, 6.74, 22.74]
    )

    # Agregar ruido
    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)

    # Takens embedding 2D
    X_emb = embedder_ts_2D.fit_transform(ts[0])

    # Calcular persistencia con Ripser
    dgms = ripser(X_emb, maxdim=1)['dgms']

    # Graficar
    ax = axs[i]

    ax.xaxis.get_offset_text().set_fontsize(5)
    ax.yaxis.get_offset_text().set_fontsize(5)
    ax.xaxis.get_offset_text().set_alpha(0.5)
    ax.yaxis.get_offset_text().set_alpha(0.5)

    colors = ['blue', 'red']
    labels = ['$H_{0}$', '$H_{1}$']
    max_val = 0

    for dim, diagram in enumerate(dgms):
        diagram = diagram[~np.isinf(diagram[:, 1])]
        if diagram.size == 0:
            continue
        ax.scatter(diagram[:, 0], diagram[:, 1],
                   color=colors[dim], label=labels[dim],
                   alpha=0.7, s=12)
        max_val = max(max_val, diagram.max())

    # línea diagonal
    ax.plot([0, max_val], [0, max_val], 'k--', lw=0.6)

    ax.set_xlabel("Birth", fontsize=6, labelpad=1)
    ax.set_ylabel("Death", fontsize=6, labelpad=1)
    ax.tick_params(axis='both', pad=1, labelsize=6)
    ax.legend(fontsize=6, loc='lower right', borderpad=0.2,
              labelspacing=0.2, handlelength=1, handletextpad=0.3, framealpha=0.6)
    ax.set_title(f"rho = {rho:.2f}", fontsize=6, pad=1)
    ax.grid(alpha=0.25)
    ax.set_aspect("equal")

plt.tight_layout()
plt.subplots_adjust(hspace=0.3, wspace=0.01)
plt.savefig(f"figure_lorenz_persistence_diagrams_rho_20-30_m={embedding_dimension_2D}_tau={embedding_time_delay_2D}.pdf",
            bbox_inches="tight", dpi=600)
plt.show()

# Reacción BZ

Este notebook calcula diagramas de persistencia (H₀ y H₁) para el oscilador de Belousov-Zhabotinsky 2D con parámetro b ∈ [2,5] (25 valores). Usa embedding de Takens (d=2, τ=57, stride=5) y Ripser. Ruido del 0%, ajustable. Visualización en subplots. Requiere gtda, ripser, teaspoon y matplotlib.

In [ ]:
############################################################
#     SUBPLOTS DE DIAGRAMAS DE PERSISTENCIA (2D)       #
############################################################

from gtda.time_series import SingleTakensEmbedding
from ripser import ripser
import matplotlib.pyplot as plt
import numpy as np
import teaspoon.MakeData.DynSysLib.DynSysLib as DSL

# --- valores del parámetro b ---
b_values = np.linspace(2, 5, 25)

embedding_dimension_2D = 2
embedding_time_delay_2D = 57
stride_2D = 5

# --- Configuración de Takens 2D ---
embedder_ts_2D = SingleTakensEmbedding(
    parameters_type="fixed",
    time_delay=embedding_time_delay_2D,
    dimension=embedding_dimension_2D,
    stride=stride_2D,
    n_jobs=2
)

# Configuración de estilo
plt.rcParams['figure.figsize'] = (12, 6)

# --- preparar figure ---
fig, axs = plt.subplots(5, 5, figsize=(13, 10))
axs = axs.flatten()

ruido = 0.02

def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
    if seed is not None:
        np.random.seed(seed)
    ts = np.asarray(ts)
    if mode == "relative":
        scale = noise_level * np.std(ts, axis=-1, keepdims=True)
    else:
        scale = noise_level
    noise = scale * np.random.randn(*ts.shape)
    return ts + noise

for i, b in enumerate(b_values):
    # Simular oscilador Belousov-Zhabotinsky 2D
    parameters = [10, b]
    t, ts = DSL.DynamicSystems(
        system='oscilador_Belousov_Zhabotinski_2D',
        dynamic_state=None,
        L=80,
        fs=90,
        SampleSize=4000,
        UserGuide=False,
        parameters=parameters,
        InitialConditions=[2, 5.1]
    )

    # Agregar ruido
    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)

    # Takens embedding 2D
    X_emb = embedder_ts_2D.fit_transform(ts[0])

    # Calcular persistencia con Ripser
    dgms = ripser(X_emb, maxdim=1)['dgms']

    # Graficar
    ax = axs[i]

    ax.xaxis.get_offset_text().set_fontsize(5)
    ax.yaxis.get_offset_text().set_fontsize(5)
    ax.xaxis.get_offset_text().set_alpha(1)
    ax.yaxis.get_offset_text().set_alpha(1)

    colors = ['blue', 'red']
    labels = ['$H_{0}$', '$H_{1}$']
    max_val = 0

    for dim, diagram in enumerate(dgms):
        diagram = diagram[~np.isinf(diagram[:, 1])]
        if diagram.size == 0:
            continue
        ax.scatter(diagram[:, 0], diagram[:, 1],
                   color=colors[dim], label=labels[dim],
                   alpha=0.7, s=12)
        max_val = max(max_val, diagram.max())

    # línea diagonal
    ax.plot([0, max_val], [0, max_val], 'k--', lw=0.6)

    ax.set_xlabel("Birth", fontsize=6, labelpad=1)
    ax.set_ylabel("Death", fontsize=6, labelpad=1)
    ax.tick_params(axis='both', pad=1, labelsize=6)
    ax.legend(fontsize=6, loc='lower right', borderpad=0.2,
              labelspacing=0.2, handlelength=1, handletextpad=0.3, framealpha=0.6)
    ax.set_title(f"b = {b:.2f}", fontsize=6, pad=1)
    ax.grid(alpha=0.25)
    ax.set_aspect("equal")

plt.tight_layout()
plt.subplots_adjust(hspace=0.3, wspace=0.01)
plt.savefig(f"figure_bz_persistence_diagrams_b_2-5_m={embedding_dimension_2D}_tau={embedding_time_delay_2D}.pdf",
            bbox_inches="tight", dpi=600)
plt.show()